# Stage 2 - Instruction Fine-Tuning (SFT)

**Base model:** Llama-3.2-3B-Instruct (4-bit / QLoRA), using the **Llama-3 chat template**.

We teach the model to answer support questions with 100+ instruction/response
pairs. We use `get_chat_template` to format conversations and
`train_on_responses_only` so the loss is computed on the **assistant answer only**
(the correct, standard SFT recipe - this is what makes training actually learn).

**Input:** `data/instruction_dataset.jsonl`
**Output:** LoRA adapter -> `models/sft_adapter/`

In [ ]:
%%capture
# Clean, version-consistent install. Let Unsloth pin its own compatible
# transformers/trl/peft versions - do NOT force-upgrade trl separately.
!pip install --upgrade unsloth unsloth_zoo
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# Get the project code + datasets into Colab.
# Replace <your-username> with your GitHub username.
!git clone https://github.com/<your-username>/customer-support-assistant.git
%cd customer-support-assistant

import os
DATA = "data"
MODELS = "models"
os.makedirs(MODELS, exist_ok=True)
print("cwd:", os.getcwd())

## 1. Load Llama-3.2-3B-Instruct + attach LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch, os

MAX_SEQ_LENGTH = 2048
BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"   # Llama-3.2-3B-Instruct, 4-bit

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,            # auto: fp16 on T4, bf16 on Ampere+
    load_in_4bit = True,     # QLoRA
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # if loss ever stays flat, try = True
    random_state = 3407,
)

## 2. Apply the Llama-3 chat template and format the dataset

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

tokenizer = get_chat_template(tokenizer, chat_template = "llama-3.1")  # works for 3.2

def formatting(examples):
    convos = [
        [{"role": "user", "content": i}, {"role": "assistant", "content": r}]
        for i, r in zip(examples["instruction"], examples["response"])
    ]
    texts = [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in convos
    ]
    return {"text": texts}

ds = load_dataset("json", data_files="data/instruction_dataset.jsonl", split="train")
ds = ds.map(formatting, batched=True)
print(">>> examples:", len(ds))     # expect 104
print(ds[0]["text"][:500])

## 3. Train (SFT, loss on assistant response only)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        packing = False,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_stage2",
        report_to = "none",
    ),
)

# Mask everything except the assistant turn so the model learns to ANSWER.
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

stats = trainer.train()
print("final (mean) loss:", stats.training_loss)   # should fall toward ~0.5-1.0

## 4. Test the instruction-tuned model (same session, no reload)

In [ ]:
FastLanguageModel.for_inference(model)

def answer(q, n=200):
    msgs = [{"role": "user", "content": q}]
    ids = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    out = model.generate(input_ids=ids, max_new_tokens=n,
                         temperature=0.7, top_p=0.9, do_sample=True,
                         eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

for q in ["How can I cancel my order?",
          "I received a damaged item. What should I do?",
          "How do I reset my password?"]:
    print("Q:", q); print("A:", answer(q)); print("-"*70)

## 5. Save the SFT adapter

In [ ]:
model.save_pretrained("models/sft_adapter")
tokenizer.save_pretrained("models/sft_adapter")
print("Saved -> models/sft_adapter")

### Stage 2 done
Fill in `reports/sft_model_comparison.md` with base-vs-SFT answers, then run
`dpo_alignment.ipynb`.